# Payment processor backward compatibility

This notebook checks that the `payment_processor_change` branches stay backward compatible with old deployment artifacts.

The code change adds new `PaymentPatternsAggregatorV2` options such as `missing_data_chars` and `use_missing_characters_and_string_length_for_trended_features`. Old parser assets and old pickled pipelines do not carry those fields, so backward compatibility means two things:

1. old parser-asset steps still run after installing the new `feature-engine-parts` / `model-engine` code; and
2. the payment-processor columns they produce match the already-saved `NORMALIZED` parquet for the same trade row.

The important ordering is `DateDiffV2` first, then `PaymentPatternsAggregatorV2`: because the payment processor has `report_date='rptDate'`, it expects `months_since_rptDate` to already exist.


In [28]:
# Install BOTH packages from the payment_processor_change branches.
# git+ssh because the Katlean repos are private and this box authenticates
# to GitHub over SSH (no HTTPS credentials). --no-deps keeps the rest of the
# env untouched. RESTART THE KERNEL after this cell the first time you run it.
%pip install --no-deps --force-reinstall \
    "git+ssh://git@github.com/Katlean/feature-engine-parts.git@payment_processor_change" \
    "git+ssh://git@github.com/Katlean/model-engine.git@payment_processor_change"


Looking in indexes: http://zamlpkgs-nginx/artifactory/api/pypi/zest_pypi/simple/
  Cloning ssh://****@github.com/Katlean/feature-engine-parts.git (to revision payment_processor_change) to /tmp/pip-req-build-s5kea9w9
  Running command git clone --filter=blob:none --quiet 'ssh://****@github.com/Katlean/feature-engine-parts.git' /tmp/pip-req-build-s5kea9w9
  Running command git checkout -b payment_processor_change --track origin/payment_processor_change
  Switched to a new branch 'payment_processor_change'
  branch 'payment_processor_change' set up to track 'origin/payment_processor_change'.
  Resolved ssh://****@github.com/Katlean/feature-engine-parts.git to commit a9bc5be74363fdee14ee1f76fd03f0631d747cae
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning ssh://****@github.com/Katlean/model-engine.git (to revision payment_processor_change) to /tmp/pip-req-build-1lt6j4t5
  Running command git clo

## Load the old deployment artifacts and comparison data

The parser asset and pipeline are old artifacts. The loaded parquet data gives us the raw/preprocessed trade row and the saved normalized output for the same `ZEST_KEY`.

We deliberately pick a `ZEST_KEY` that occurs exactly once (a single-tradeline applicant, so the comparison is one clean row) and that has a `*` in `PAYMENT_HISTORY_1_24` (the most recent 24 months for equifax cms_6). The `*` is the missing-data character, so choosing such a row makes the check meaningful: it runs the old asset through the new feature-engine-parts on data that actually exercises the missing-data-char path, and confirms the new code still reproduces the saved normalized output. If the new behavior had accidentally become the default, this row's payment-processor columns would diverge from normalized and we would catch it. We are picking a `*` row on purpose to make sure backward compatibility is actually working, not just trivially passing on rows with no missing data.

## **NOTE WE CAN SEE BELOW THAT MODEL_ENGINE IS UPDATED AND SO IS FE PARTS**

In [34]:
from model_engine.assets.utils import load_asset
load_asset('equifax/cms_6/fe2/trade.json')['preprocess'][3]

{'type': 'PaymentPatternsAggregatorV2',
 'params': {'report_date': 'rptDate',
  'missing_data_chars': ['*'],
  'use_missing_characters_and_string_length_for_trended_features': True,
  'payment_patterns': {'patterns': ['RATE_STATUS_CODE',
    'PAYMENT_HISTORY_1_24',
    'PAYMENT_HISTORY_25_36',
    'PAYMENT_HISTORY_37_48'],
   'rate': {'paid_as_agreed': ['0', '1'],
    'DQ30+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ60+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ90+': ['4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ120+': ['5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'CO': ['6', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ30': ['2'],
    'DQ60': ['3'],
    'DQ90': ['4'],
    'DQ120': ['5']},
   'trim': 48,
   'placeholder': '/',
   'keep': ['DQ30+',
    'DQ60+',
    'DQ90+',
    'DQ120+',
    'CO',
    'DQ30',
    'DQ60',
    'DQ90']}},
 'notes': "missing_data_chars: '*' = rate/status not available for that month (Equifax Sy

In [36]:
from feature_engine_parts.fe_parts_V2.preprocessors.payment_pattern_aggregator import PaymentPatternsAggregatorV2

import inspect 

print(inspect.getsource(PaymentPatternsAggregatorV2.transform))

    def transform(self, data):
        data = data.copy(deep=True)
        input_columns = list(data.columns)
        # read defensively: pipelines pickled before these attributes existed
        # won't have them, and unpickling does not re-run __init__.
        missing_data_chars = getattr(self, "missing_data_chars", ["*"])
        use_missing_characters_and_string_length_for_trended_features = getattr(
            self, "use_missing_characters_and_string_length_for_trended_features", False
        )
        data[self.zest_ppt_name] = self._construct_payment_pattern_cols(data)
        data[self.ppt_len_name] = self._get_effective_month_range(
            data[self.zest_ppt_name], data[self.zest_ppt_name].str.len(), missing_data_chars=missing_data_chars
        )
        data = self._construct_trended_features(
            data,
            data[self.zest_ppt_name],
            missing_data_chars=missing_data_chars,
            use_missing_characters_and_string_length_for_trended_feat

In [1]:
import copy
import json

import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype

from feature_engine_parts.factory import load_object
from zaml.common.utils.io import load_state


def load_json(path):
    with open(path) as f:
        return json.load(f)


/home/jag/.conda/envs/model_engine_2_py310/lib/python3.10/site-packages/zaml/common/utils/io.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
PARSER_ASSET_PATH = '/d/shared/silver_projects_v2/californiacu/autoloanCreditcardHomeequityPersonalloanv1/deployment/autoloan/model1_AppData_LTVAutoloan/meridianlinkv1/parser_asset.json'
PIPELINE_PATH = '/d/shared/silver_projects_v2/californiacu/autoloanCreditcardHomeequityPersonalloanv1/modeling/model_artifacts/autoloan/model1_AppData_LTVAutoloan/pipeline.obj'

PREPROCESSED_S3 = 's3://power-client-data-staging/PREPROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=trade/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/ARCHIVE_DATE=2021-06-30/'
NORMALIZED_S3 = 's3://power-client-data-staging/NORMALIZED/DATA/TABLE=trade/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/ARCHIVE_DATE=2021-06-30'

# TEST_ZEST_KEY is selected in the next cell, after loading: a row that
# occurs exactly once and has a '*' in PAYMENT_HISTORY_1_24 (the most
# recent 24 months for equifax cms_6).


In [3]:
parser_asset = load_json(PARSER_ASSET_PATH)

pre_processed_data = pd.read_parquet(PREPROCESSED_S3)
normalized_data = pd.read_parquet(NORMALIZED_S3)

# Pick a ZEST_KEY that occurs exactly once in BOTH tables (a single-tradeline
# applicant, so the comparison is one clean row) AND has a '*' in
# PAYMENT_HISTORY_1_24 (the most recent 24 months for equifax cms_6). The '*'
# means the row exercises the missing-data-char path, so this is a real
# backward-compat check: the new code on the old asset must still match the
# saved normalized output.
pre_counts = pre_processed_data['ZEST_KEY'].value_counts()
norm_counts = normalized_data['ZEST_KEY'].value_counts()
single_keys = set(pre_counts[pre_counts == 1].index) & set(norm_counts[norm_counts == 1].index)

DQ30_CODES = ["2", "3", "4", "5", "6", "7", "8", "9", "G", "K", "L", "Z"]
hist = pre_processed_data['PAYMENT_HISTORY_1_24'].fillna('')
has_star = hist.str.contains('*', regex=False)
# any one of the DQ30+ codes anywhere in the recent window (regex alternation)
has_dq30 = hist.str.contains('|'.join(DQ30_CODES), regex=True)
candidates = pre_processed_data.loc[
    has_star & has_dq30 & pre_processed_data['ZEST_KEY'].isin(single_keys), 'ZEST_KEY'
].tolist()
assert candidates, "no single-occurrence ZEST_KEY with a '*' and a DQ30+ code in PAYMENT_HISTORY_1_24"
TEST_ZEST_KEY = candidates[0]
print('selected ZEST_KEY:', TEST_ZEST_KEY)

pre_processed_data_filtered = pre_processed_data[pre_processed_data['ZEST_KEY'] == TEST_ZEST_KEY].copy()
normalized_data_filtered = normalized_data[normalized_data['ZEST_KEY'] == TEST_ZEST_KEY].copy()

print('pre_processed rows:', len(pre_processed_data_filtered), '| normalized rows:', len(normalized_data_filtered))
print('PAYMENT_HISTORY_1_24:', pre_processed_data_filtered['PAYMENT_HISTORY_1_24'].iloc[0])
assert len(pre_processed_data_filtered) == len(normalized_data_filtered) == 1


/home/jag/.conda/envs/model_engine_2_py310/lib/python3.10/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


selected ZEST_KEY: 212974_1_276_12
pre_processed rows: 1 | normalized rows: 1
PAYMENT_HISTORY_1_24: ***555554321/11111*******


In [4]:
pre_processed_data_filtered

,ZEST_KEY,quarter,appDate,SEG_SEQ,SEG_PARENT,SEG_PARENT_SEQ,SEGMENT_TYPE,CUSTOMER_NAME,CUSTOMER_NUMBER,DATE_REPORTED,...,PREVIOUS_HIGH_RATE_BEFORE_HISTORY,PREVIOUS_HIGH_DATE_BEFORE_HISTORY,NARRATIVE_CODE_1X,NARRATIVE_CODE_2X,NARRATIVE_CODE_3X,NARRATIVE_CODE_4X,RECENT_TRADE_FLAG,QUALIFYING_FLAG,ARCHIVE_DATA,DATE_OF_REQUEST
20954,212974_1_276_12,2021Q2,2021-07-21,1,FULL-Header,1,PT,None,AU,06302020,...,*,None,098,132,None,None,Y,Y,None,2021-06-29


In [5]:
normalized_data_filtered

,ZEST_KEY,date_of_request,balance_amt,credit_limit,high_credit_amt,pastDueAmt,openDate,closedDate,rptDate,lstPmtDate,...,has_utilization_over_25_percent,has_utilization_over_50_percent,has_utilization_over_100_percent,has_historic_utilization_over_25_percent,has_historic_utilization_over_50_percent,has_historic_utilization_over_100_percent,has_balance_greater_than_hc,is_individual,is_joint,is_joint_not_mortgage
20954,212974_1_276_12,06292021,0.0,NaN,2510.0,0.0,12272018,04012020,06302020,04012020,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [6]:
pre_processed_data_filtered[[
    'ZEST_KEY',
    'DATE_OF_REQUEST',
    'DATE_REPORTED',
    'RATE_STATUS_CODE',
    'PAYMENT_HISTORY_1_24',
    'PAYMENT_HISTORY_25_36',
    'PAYMENT_HISTORY_37_48',
]]


,ZEST_KEY,DATE_OF_REQUEST,DATE_REPORTED,RATE_STATUS_CODE,PAYMENT_HISTORY_1_24,PAYMENT_HISTORY_25_36,PAYMENT_HISTORY_37_48
20954,212974_1_276_12,2021-06-29,06302020,5,***555554321/11111*******,/************,/************


## Shared comparison helpers

These helpers run FE-parts objects from old asset dictionaries and compare only the columns created by `PaymentPatternsAggregatorV2` against the saved normalized parquet.


In [7]:
def fe_part_from_parser_step(step):
    # Instantiate a feature-engine-parts object from one parser-asset step.
    return load_object({'type': step['type'], 'params': copy.deepcopy(step.get('params', {}))})


def payment_processor_columns(payment_transformer, expected_normalized):
    # Return payment-processor output columns that exist in the normalized parquet.
    cols = []

    try:
        cols.extend(payment_transformer.output_features)
    except Exception:
        pass

    if not cols:
        cols.extend(getattr(payment_transformer, '_new_features', []))
        if getattr(payment_transformer, 'return_zest_ppt', False):
            cols.append(getattr(payment_transformer, 'zest_ppt_name', 'zest_payment_pattern'))
        if getattr(payment_transformer, 'return_ppt_len', False):
            cols.append(getattr(payment_transformer, 'ppt_len_name', 'payment_history_length'))

    return [c for c in dict.fromkeys(cols) if c in expected_normalized.columns]


def compare_payment_processor_outputs(candidate, expected, columns, label, atol=1e-6):
    candidate = candidate.reset_index(drop=True)
    expected = expected.reset_index(drop=True)
    assert len(candidate) == len(expected), f'row-count mismatch: {len(candidate)} vs {len(expected)}'

    rows = []
    for col in columns:
        left = candidate[col]
        right = expected[col]

        if is_numeric_dtype(left) or is_numeric_dtype(right):
            left_values = pd.to_numeric(left, errors='coerce').to_numpy(dtype='float64')
            right_values = pd.to_numeric(right, errors='coerce').to_numpy(dtype='float64')
            matches = np.isclose(left_values, right_values, rtol=0, atol=atol, equal_nan=True)
        else:
            left_values = left.astype('string')
            right_values = right.astype('string')
            eq = left_values.eq(right_values).fillna(False)
            both_na = left_values.isna() & right_values.isna()
            matches = (eq | both_na).to_numpy(dtype=bool)

        # ensure a real numpy bool array (avoids ~True == -2 on object arrays)
        matches = np.asarray(matches, dtype=bool)
        mismatch_idx = np.flatnonzero(~matches)
        # always show a row's values per column so they can be eyeballed:
        # the first mismatching row if any, otherwise the first row.
        show_idx = int(mismatch_idx[0]) if len(mismatch_idx) else 0
        rows.append({
            'column': col,
            'matches': int(matches.sum()),
            'mismatches': int((~matches).sum()),
            'candidate': left.iloc[show_idx] if len(left) else None,
            'normalized': right.iloc[show_idx] if len(right) else None,
        })

    summary = pd.DataFrame(rows).sort_values(['mismatches', 'column'], ascending=[False, True])
    total_mismatches = int(summary['mismatches'].sum()) if len(summary) else 0
    print(f'{label}: {len(columns)} columns checked; {total_mismatches} mismatched cells')
    return summary


## Parser-asset transformer path

This path uses the trade transformer list from `parser_asset.json`. We run the flattened feature-engine-parts steps from the raw/preprocessed row through `PaymentPatternsAggregatorV2`, so the mapped date fields and `months_since_rptDate` are created before the payment processor is called.


In [8]:
trade_input_normalizer = next(
    step for step in parser_asset['etl_steps'][1]['input_normalizer_steps']
    if step['source_name'] == 'trade'
)

In [9]:
trade_input_normalizer.keys()

dict_keys(['source_name', 'input_name', 'fe_version', 'transformers'])

In [10]:

trade_transformers = trade_input_normalizer['transformers']

In [11]:
trade_transformers[0]

{'type': 'InsertMissingColumnsV2',
 'params': {'columns': {'ZEST_KEY': 'str',
   'DATE_OF_REQUEST': 'str',
   'BALANCE': 'str',
   'CREDIT_LIMIT': 'str',
   'HIGH_CREDIT': 'str',
   'PAST_DUE_AMOUNT': 'str',
   'SCHEDULED_PAYMENT_AMOUNT': 'str',
   'DATE_OPENED': 'str',
   'CLOSED_DATE': 'str',
   'DATE_REPORTED': 'str',
   'DMD_REPORTED': 'str',
   'LAST_PAYMENT_DATE': 'str',
   'PREVIOUS_HIGH_DATE_1': 'str',
   'TERMS_FREQUENCY': 'str',
   'TERMS_DURATION': 'str',
   'PAYMENT_HISTORY_1_24': 'str',
   'PAYMENT_HISTORY_25_36': 'str',
   'PAYMENT_HISTORY_37_48': 'str',
   'RATE_STATUS_CODE': 'str',
   'ACCOUNT_TYPE': 'str',
   'PORTFOLIO_TYPE': 'str',
   'ECOA_DESIGNATOR': 'str',
   'NARRATIVE_CODE_1': 'str',
   'NARRATIVE_CODE_2': 'str',
   'ACTIVITY_DESIGNATOR': 'str',
   'ORIGINAL_CHARGE_OFF_AMOUNT': 'str',
   'CUSTOMER_NUMBER': 'str'},
  'fill_with': nan,
  'dtype': 'string',
  'transformer_name': None},
 'attributes': {},
 'fe_parts_version': '2.0.1'}

In [12]:

months_since_rpt_date = next(
    step for step in trade_transformers
    if step['type'] == 'DateDiffV2'
    and step.get('params', {}).get('new_feature') == 'months_since_rptDate'
)

In [13]:
months_since_rpt_date

{'type': 'DateDiffV2',
 'params': {'feature': 'rptDate',
  'reference_feature': 'date_of_request',
  'new_feature': 'months_since_rptDate',
  'units': 'M',
  'transformer_name': 'months_since_rptDate'},
 'attributes': {},
 'fe_parts_version': '2.0.1'}

In [14]:

trade_transformer = next(
    step for step in trade_transformers
    if step['type'] == 'PaymentPatternsAggregatorV2'
)

In [15]:
trade_transformer

{'type': 'PaymentPatternsAggregatorV2',
 'params': {'payment_patterns': {'patterns': ['RATE_STATUS_CODE',
    'PAYMENT_HISTORY_1_24',
    'PAYMENT_HISTORY_25_36',
    'PAYMENT_HISTORY_37_48'],
   'rate': {'paid_as_agreed': ['0', '1'],
    'DQ30+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ60+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ90+': ['4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ120+': ['5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'CO': ['6', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ30': ['2'],
    'DQ60': ['3'],
    'DQ90': ['4'],
    'DQ120': ['5']},
   'trim': 48,
   'placeholder': '/',
   'keep': ['DQ30+',
    'DQ60+',
    'DQ90+',
    'DQ120+',
    'CO',
    'DQ30',
    'DQ60',
    'DQ90']},
  'report_date': 'rptDate',
  'month_ranges': [3, 6, 12, 24, 48],
  'name': '',
  'return_zest_ppt': True,
  'return_ppt_len': True,
  'transformer_name': 'PaymentPatternsAggregatorV2'},
 'attributes': {},
 'fe_parts_versi

In [16]:
months_since_step_idx = trade_transformers.index(months_since_rpt_date)

In [17]:
months_since_step_idx

31

In [18]:
payment_step_idx = trade_transformers.index(trade_transformer)

In [19]:
payment_step_idx

34

In [20]:
print('months_since_rptDate step:', months_since_step_idx, months_since_rpt_date)
print('payment processor step:', payment_step_idx, trade_transformer)

months_since_rptDate step: 31 {'type': 'DateDiffV2', 'params': {'feature': 'rptDate', 'reference_feature': 'date_of_request', 'new_feature': 'months_since_rptDate', 'units': 'M', 'transformer_name': 'months_since_rptDate'}, 'attributes': {}, 'fe_parts_version': '2.0.1'}
payment processor step: 34 {'type': 'PaymentPatternsAggregatorV2', 'params': {'payment_patterns': {'patterns': ['RATE_STATUS_CODE', 'PAYMENT_HISTORY_1_24', 'PAYMENT_HISTORY_25_36', 'PAYMENT_HISTORY_37_48'], 'rate': {'paid_as_agreed': ['0', '1'], 'DQ30+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'], 'DQ60+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'], 'DQ90+': ['4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'], 'DQ120+': ['5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'], 'CO': ['6', '8', '9', 'G', 'K', 'L', 'Z'], 'DQ30': ['2'], 'DQ60': ['3'], 'DQ90': ['4'], 'DQ120': ['5']}, 'trim': 48, 'placeholder': '/', 'keep': ['DQ30+', 'DQ60+', 'DQ90+', 'DQ120+', 'CO', 'DQ30', 'DQ60', 'DQ90']}, 'report_date'

In [21]:







assert months_since_step_idx < payment_step_idx


In [22]:
def run_parser_trade_steps_through_payment(data, steps):
    out = data.copy()
    ran_steps = []
    payment_transformer = None

    for idx, step in enumerate(steps):
        transformer = fe_part_from_parser_step(step)
        out = transformer.transform(out)
        ran_steps.append((idx, step['type'], getattr(transformer, 'transformer_name', None)))

        if step['type'] == 'PaymentPatternsAggregatorV2':
            payment_transformer = transformer
            break

    assert payment_transformer is not None, 'PaymentPatternsAggregatorV2 was not found in the parser trade steps'
    return out, payment_transformer, ran_steps


parser_payment_output, parser_payment_transformer, parser_steps_ran = run_parser_trade_steps_through_payment(
    pre_processed_data_filtered,
    trade_transformers,
)

print('ran steps:', len(parser_steps_ran))
parser_payment_output[[
    'ZEST_KEY',
    'date_of_request',
    'rptDate',
    'months_since_rptDate',
    'zest_payment_pattern',
    'payment_history_length',
]]


ran steps: 35


,ZEST_KEY,date_of_request,rptDate,months_since_rptDate,zest_payment_pattern,payment_history_length
20954,212974_1_276_12,2021-06-29,2020-06-30,11.959178,############5***55555432111111****************...,18.0


In [23]:
parser_payment_columns = payment_processor_columns(parser_payment_transformer, normalized_data_filtered)
assert parser_payment_columns, 'No parser payment output columns were found in normalized_data_filtered'
parser_payment_comparison = compare_payment_processor_outputs(
    parser_payment_output,
    normalized_data_filtered,
    parser_payment_columns,
    label='parser_asset payment processor vs normalized parquet',
)

assert parser_payment_comparison['mismatches'].sum() == 0
parser_payment_comparison


parser_asset payment processor vs normalized parquet: 90 columns checked; 0 mismatched cells


,column,matches,mismatches,candidate,normalized
54,months_since_most_recent_CO,1,0,NaN,NaN
43,months_since_most_recent_DQ120+,1,0,12.0,12.0
65,months_since_most_recent_DQ30,1,0,23.0,23.0
10,months_since_most_recent_DQ30+,1,0,12.0,12.0
76,months_since_most_recent_DQ60,1,0,22.0,22.0
...,...,...,...,...,...
84,percent_DQ90_24_months,1,0,0.083333,0.083333
78,percent_DQ90_3_months,1,0,NaN,NaN
86,percent_DQ90_48_months,1,0,0.027778,0.027778
80,percent_DQ90_6_months,1,0,NaN,NaN


In [24]:
parser_payment_comparison[parser_payment_comparison['mismatches']!=0]

,column,matches,mismatches,candidate,normalized


## Loaded pipeline transformer path

This path uses the old pickled pipeline object. We use the pipeline's own mapper to create normalized payment inputs from the raw/preprocessed row, then call the loaded `months_since_rptDate` transformer followed by the loaded `PaymentPatternsAggregatorV2` transformer.


In [25]:
pipeline = load_state(PIPELINE_PATH)
node_dict = {node.name: node for node in pipeline.graph.nodes}

trade_fe = node_dict['trade FE'].transformer
pipeline_trade_fe_asset = trade_fe.asset
input_normalizer = trade_fe.transformers['InputNormalizer']
preprocessors = input_normalizer.transformers['Preprocessors']

transformer_months_since_rpt_date = preprocessors.transformers['months_since_rptDate']
transformer_payment_processor = preprocessors.transformers['PaymentPatternsAggregatorV2']

print('InputNormalizer steps:', list(input_normalizer.transformers.keys()))
print('first preprocessors:', list(preprocessors.transformers.keys())[:5])
print('loaded month transformer:', transformer_months_since_rpt_date)
print('loaded payment transformer:', transformer_payment_processor)


/home/jag/.conda/envs/model_engine_2_py310/lib/python3.10/site-packages/zaml/common/utils/io.py:59: UserWarning: ZAML version mismatch: current enviroment is on ZAML 34.5.3
                      but the loaded object is built on ZAML 34.5.4. Loading serialized
                      object from different versions are not recommended and may yield unexpected
                      errors or results.
  warnings.warn(
/home/jag/.conda/envs/model_engine_2_py310/lib/python3.10/site-packages/zaml/common/utils/io.py:96: UserWarning: Dependency version mismatch: These dependencies have different versions when the object was built
                      (name, curr_ver, org_ver): [('ipython', '8.38.0', '8.30.0'), ('ipywidgets', '8.1.8', '7.8.5'), ('jupyter-book', '2.1.2', '0.15.1'), ('kaleido', '1.2.0', '0.2.1'), ('scikit-learn', '1.2.2', '1.2.1'), ('Sphinx', '8.0.2', '5.0.2'), ('sphinx-rtd-theme', '3.1.0', '1.3.0'), ('sphinxcontrib-bibtex', '2.6.1', '2.5.0')]. This might yield unexpected result i

InputNormalizer steps: ['Mappers', 'Preprocessors', 'Postprocessors']
first preprocessors: ['months_since_rptDate', 'months_since_openDate', 'months_since_lstPmtDate', 'PaymentPatternsAggregatorV2', 'high_credit_amt']
loaded month transformer: <feature_engine_parts.fe_parts_V2.preprocessors.date_diff.DateDiffV2 object at 0x7f34ec19e290>
loaded payment transformer: <feature_engine_parts.fe_parts_V2.preprocessors.payment_pattern_aggregator.PaymentPatternsAggregatorV2 object at 0x7f34ec19d570>


In [26]:
date_diff_asset_pipe = pipeline_trade_fe_asset['InputNormalizer']['preprocess'][0]
payment_patterns_asset_pipe = pipeline_trade_fe_asset['InputNormalizer']['preprocess'][3]

date_diff_asset_pipe, payment_patterns_asset_pipe


({'type': 'DateDiffV2',
  'params': {'feature': 'rptDate',
   'reference_feature': 'date_of_request',
   'new_feature': 'months_since_rptDate'}},
 {'type': 'PaymentPatternsAggregatorV2',
  'params': {'report_date': 'rptDate',
   'payment_patterns': {'patterns': ['RATE_STATUS_CODE',
     'PAYMENT_HISTORY_1_24',
     'PAYMENT_HISTORY_25_36',
     'PAYMENT_HISTORY_37_48'],
    'rate': {'paid_as_agreed': ['0', '1'],
     'DQ30+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
     'DQ60+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
     'DQ90+': ['4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
     'DQ120+': ['5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
     'CO': ['6', '8', '9', 'G', 'K', 'L', 'Z'],
     'DQ30': ['2'],
     'DQ60': ['3'],
     'DQ90': ['4'],
     'DQ120': ['5']},
    'trim': 48,
    'placeholder': '/',
    'keep': ['DQ30+',
     'DQ60+',
     'DQ90+',
     'DQ120+',
     'CO',
     'DQ30',
     'DQ60',
     'DQ90']}}})

In [27]:
pipeline_payment_input = input_normalizer.transformers['Mappers'].transform(pre_processed_data_filtered)
pipeline_payment_input = transformer_months_since_rpt_date.transform(pipeline_payment_input)
pipeline_payment_output = transformer_payment_processor.transform(pipeline_payment_input)

pipeline_payment_output[[
    'ZEST_KEY',
    'date_of_request',
    'rptDate',
    'months_since_rptDate',
    'zest_payment_pattern',
    'payment_history_length',
]]


,ZEST_KEY,date_of_request,rptDate,months_since_rptDate,zest_payment_pattern,payment_history_length
20954,212974_1_276_12,2021-06-29,2020-06-30,11.959178,############5***55555432111111****************...,18.0


In [28]:
pipeline_payment_columns = payment_processor_columns(transformer_payment_processor, normalized_data_filtered)
assert pipeline_payment_columns, 'No pipeline payment output columns were found in normalized_data_filtered'
pipeline_payment_comparison = compare_payment_processor_outputs(
    pipeline_payment_output,
    normalized_data_filtered,
    pipeline_payment_columns,
    label='pickled pipeline payment processor vs normalized parquet',
)
assert pipeline_payment_comparison['mismatches'].sum() == 0
pipeline_payment_comparison


pickled pipeline payment processor vs normalized parquet: 90 columns checked; 0 mismatched cells


,column,matches,mismatches,candidate,normalized
54,months_since_most_recent_CO,1,0,NaN,NaN
43,months_since_most_recent_DQ120+,1,0,12.0,12.0
65,months_since_most_recent_DQ30,1,0,23.0,23.0
10,months_since_most_recent_DQ30+,1,0,12.0,12.0
76,months_since_most_recent_DQ60,1,0,22.0,22.0
...,...,...,...,...,...
84,percent_DQ90_24_months,1,0,0.083333,0.083333
78,percent_DQ90_3_months,1,0,NaN,NaN
86,percent_DQ90_48_months,1,0,0.027778,0.027778
80,percent_DQ90_6_months,1,0,NaN,NaN


## Same row, new behavior: the response changes

Everything above confirms backward compatibility: the loaded pipeline aggregator,
run as-is, reproduces the normalized parquet exactly (0 mismatches). That is the
old behavior, because the pickled object predates the new flag so `getattr` resolves
`use_missing_characters_and_string_length_for_trended_features` to `False`.

Here we keep *everything* the same - the same selected row, the same loaded
`PaymentPatternsAggregatorV2` object, the same `pipeline_payment_input` - and only
flip the new flag on. This is exactly what the updated FE2 asset does
(`"use_missing_characters_and_string_length_for_trended_features": true`).

We deep-copy the loaded transformer so the original is untouched, set the flag to
`True`, and re-run `transform` on the identical input. The trended `percent_*`
features now divide by observed months only (string length minus the bureau's
missing-data char anywhere in the window) instead of the nominal window, so the
response differs from the normalized/old output.

In [37]:
import copy

# same loaded object, same input - only the new flag is flipped on
new_behavior_transformer = copy.deepcopy(transformer_payment_processor)
new_behavior_transformer.use_missing_characters_and_string_length_for_trended_features = True
# the updated asset declares missing_data_chars; equifax cms_6 uses ["*"]
new_behavior_transformer.missing_data_chars = getattr(
    new_behavior_transformer, 'missing_data_chars', ['*']
) or ['*']

print('flag on  :', new_behavior_transformer.use_missing_characters_and_string_length_for_trended_features)
print('missing  :', new_behavior_transformer.missing_data_chars)

# identical input row used in the pipeline path above
new_behavior_output = new_behavior_transformer.transform(pipeline_payment_input)

new_behavior_output[[
    'ZEST_KEY',
    'zest_payment_pattern',
    'payment_history_length',
]]

flag on  : True
missing  : ['*']


,ZEST_KEY,zest_payment_pattern,payment_history_length
20954,212974_1_276_12,############5***55555432111111****************...,18.0


In [38]:
# compare the NEW behavior against the normalized (old) parquet on the same columns
new_behavior_comparison = compare_payment_processor_outputs(
    new_behavior_output,
    normalized_data_filtered,
    pipeline_payment_columns,
    label='NEW behavior (flag on) vs normalized parquet',
)

# the columns that now differ - these are the trended percent_* denominators
new_behavior_comparison[new_behavior_comparison['mismatches'] != 0]

NEW behavior (flag on) vs normalized parquet: 90 columns checked; 14 mismatched cells


,column,matches,mismatches,candidate,normalized
40,percent_DQ120+_24_months,0,1,0.666667,0.5
42,percent_DQ120+_48_months,0,1,0.4,0.166667
7,percent_DQ30+_24_months,0,1,1.0,0.75
9,percent_DQ30+_48_months,0,1,0.6,0.25
62,percent_DQ30_24_months,0,1,0.111111,0.083333
64,percent_DQ30_48_months,0,1,0.066667,0.027778
18,percent_DQ60+_24_months,0,1,0.888889,0.666667
20,percent_DQ60+_48_months,0,1,0.533333,0.222222
73,percent_DQ60_24_months,0,1,0.111111,0.083333
75,percent_DQ60_48_months,0,1,0.066667,0.027778


The rows above are the features whose response changed. `candidate` is the new
behavior (observed-months denominator) and `normalized` is the old output. The
non-trended columns (counts, `months_since_*`, `zest_payment_pattern`,
`payment_history_length`) stay identical - only the trended `percent_*` features
move, which is exactly the intended effect of the fix.